# FastAPI 进阶模式 (Advanced Patterns)

> 本 Notebook 展示了 FastAPI 中的高级依赖注入、应用生命周期管理以及如何进行高效的 Mock 测试。

## 1. 依赖注入 (Depends)
演示如何通过依赖注入实现权限验证的解耦。

In [ ]:
from fastapi import FastAPI, Depends, Header, HTTPException

async def verify_key(x_key: str = Header(...)):
    if x_key != "secret":
        raise HTTPException(status_code=400, detail="Invalid Key")
    return x_key

app = FastAPI()

@app.get("/data")
async def get_data(key: str = Depends(verify_key)):
    return {"status": "access granted", "key": key}

print("FastAPI app defined with DI.")

## 2. 依赖覆盖测试
演示如何在不启动真实环境的情况下测试逻辑。

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 正常测试会失败（缺 Header）
res1 = client.get("/data")
print(f"No header status: {res1.status_code}")

# 覆盖依赖项：强制跳过校验
app.dependency_overrides[verify_key] = lambda: "mocked-key"
res2 = client.get("/data")
print(f"Overridden status: {res2.status_code}, data: {res2.json()}")

app.dependency_overrides.clear()